# aihydro-watershed — Global Capability Demo

End-to-end demonstration of **all five capability domains** on watersheds across three continents, showing that every function in `aihydro-watershed` is globally capable with no hard-coded CONUS dependencies.

| Basin | Location | Area | Delineation method |
|---|---|---|---|
| **Potomac R. (CONUS)** | Maryland, USA | ~25 000 km² | USGS NLDI gauge |
| **Thur R. (Europe)** | Switzerland | ~1 730 km² | Pour-point (MERIT) |
| **Ping R. (SE Asia)** | Thailand | ~6 500 km² | Pour-point (MERIT) |

**Five domains covered:**
1. Delineation
2. Geomorphic characterization (28 parameters)
3. Terrain intelligence (TWI, Curve Number, event runoff, RUSLE erosion)
4. CAMELS-style hydrological signatures (17 metrics)
5. Specialist signature tools (baseflow, FDC, flood frequency, drought SPI)

**Global data stack used automatically:**
- DEM: GLO-30 (global, GEE)
- Land cover: ESA WorldCover → NLCD remap (Planetary Computer STAC, no auth)
- Soil: POLARIS (CONUS) / SoilGrids (global, GEE)
- Streamflow: USGS NWIS (CONUS) / GEOGLOWS v2 → Open-Meteo GloFAS (global, no auth)
- Precipitation: GridMET (CONUS) / ERA5-Land / CHIRPS (GEE) / CHIRPS-IRI OPeNDAP (no auth fallback)


---
## 0. Setup & basin definitions


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from shapely.geometry import shape, mapping, Point

# aihydro-watershed public API
from aihydro_watershed.delineation.router import delineate_from_point
from aihydro_watershed.characterize.geomorphic import extract_geomorphic_parameters
from aihydro_watershed.characterize.twi import compute_twi_result
from aihydro_watershed.terrain.curve_number import create_curve_number_grid_from_geometry
from aihydro_watershed.terrain.event_runoff import (
    scs_cn_runoff, storage_s_mm, design_hydrograph, scs_triangular_hydrograph
)
from aihydro_watershed.terrain.erosion import (
    rusle, k_factor_epic, ls_factor_wischmeier,
    c_factor_from_landcover, r_factor_renard_freimund,
    modified_fournier_index
)
from aihydro_watershed.signatures.signatures import (
    extract_hydrological_signatures,
    compute_flow_stats_camels, compute_water_balance_camels,
    compute_event_stats_camels, compute_timing_stats_camels, compute_slope_fdc_camels
)
from aihydro_watershed.signatures.baseflow import lyne_hollick, ukih, compute_bfi
from aihydro_watershed.signatures.flow_duration import flow_duration_curve, seven_q10
from aihydro_watershed.signatures.flood_frequency import annual_maxima, flood_frequency
from aihydro_watershed.signatures.drought_indices import spi, spei

print('aihydro-watershed loaded ✅')


In [ ]:
# ── Basin definitions ────────────────────────────────────────────────────────
# Each basin: a GAUGE_ID (CONUS NWIS) or a LAT/LON pour-point (global).

BASINS = {
    'potomac': {
        'label'     : 'Potomac R. (CONUS, USA)',
        'gauge_id'  : '01638500',    # USGS gauge — used for signatures only
        'pour_lat'  : 39.2732,       # USGS gauge 01638500, Point of Rocks MD
        'pour_lon'  : -77.5418,
        'area_km2'  : 25000.0,       # approximate; overwritten by delineation result
        'start_date': '2000-01-01',
        'end_date'  : '2004-12-31',
    },
    'thur': {
        'label'     : 'Thur R. (Europe, Switzerland)',
        'gauge_id'  : None,
        'pour_lat'  : 47.60,         # Andelfingen gauging point
        'pour_lon'  : 8.68,
        'area_km2'  : 1730.0,
        'start_date': '2000-01-01',
        'end_date'  : '2004-12-31',
    },
    'ping': {
        'label'     : 'Ping R. (SE Asia, Thailand)',
        'gauge_id'  : None,
        'pour_lat'  : 18.78,         # Chiang Mai outlet
        'pour_lon'  : 98.98,
        'area_km2'  : 6500.0,
        'start_date': '2000-01-01',
        'end_date'  : '2004-12-31',
    },
}

print('Basin definitions ready.')
for k, b in BASINS.items():
    loc = f"gauge {b['gauge_id']}" if b['gauge_id'] else f"({b['pour_lat']}°N, {b['pour_lon']}°E)"
    print(f'  {b["label"]:40s}  {loc}')


---
## 1. Delineation

NLDI pour-point delineation for the CONUS gauge; MERIT-Hydro pyflwdir pipeline for global pour-points.


In [ ]:
results = {}    # store HydroResult per basin
wsheds  = {}    # GeoDataFrame per basin

for name, b in BASINS.items():
    print(f"\n{'='*55}")
    print(f"Delineating: {b['label']}")
    try:
        r = delineate_from_point(b['pour_lat'], b['pour_lon'])

        results[name] = r
        geom = shape(r.data['geometry_geojson'])
        wsheds[name] = gpd.GeoDataFrame(geometry=[geom], crs='EPSG:4326')
        BASINS[name]['area_km2']   = r.data.get('area_km2', b['area_km2'])
        BASINS[name]['outlet_lat'] = r.data.get('outlet_lat', b['pour_lat'] or 0)
        BASINS[name]['outlet_lon'] = r.data.get('outlet_lon', b['pour_lon'] or 0)
        print(f"  Area     : {r.data.get('area_km2', 0):,.0f} km²")
        print(f"  Method   : {r.data.get('method_used', '?')}")
        print(f"  Outlet   : ({r.data.get('outlet_lat','?'):.4f}°N, {r.data.get('outlet_lon','?'):.4f}°E)")
    except Exception as e:
        print(f'  ⚠️  delineation failed: {e}')


In [ ]:
# Map — all three watersheds
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, (name, b), color in zip(axes, BASINS.items(), colors):
    if name in wsheds:
        wsheds[name].plot(ax=ax, color=color, alpha=0.5, edgecolor='black', linewidth=1.2)
        cx, cy = wsheds[name].geometry.iloc[0].centroid.x, wsheds[name].geometry.iloc[0].centroid.y
        ax.set_title(b['label'], fontsize=9, fontweight='bold')
        ax.set_xlabel('Longitude (°E)')
        ax.set_ylabel('Latitude (°N)')
        ax.annotate(f"{b['area_km2']:,.0f} km²", xy=(cx, cy), ha='center', fontsize=9,
                    color='white', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc=color, alpha=0.8))
    else:
        ax.text(0.5,0.5,'delineation failed', transform=ax.transAxes, ha='center')
        ax.set_title(b['label'])

plt.suptitle('Three global watersheds — aihydro-watershed delineation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 2. Geomorphic characterization (28 parameters)

All 28 parameters from Table 4.1 of the FUG paper — extracted via DEM analysis (GLO-30 global, GEE).
Uses `pyproj.Geod` for geodesic area/perimeter and UTM zone projection per basin for accurate distance metrics.


In [ ]:
geo_params = {}

for name, b in BASINS.items():
    if name not in wsheds:
        print(f'Skipping {name} (delineation failed)')
        continue
    print(f"\nGeomorphic — {b['label']}")
    try:
        params, units = extract_geomorphic_parameters(
            wsheds[name],
            outlet_lat=b.get('outlet_lat', b.get('pour_lat', 0)),
            outlet_lon=b.get('outlet_lon', b.get('pour_lon', 0)),
        )
        geo_params[name] = {'params': params, 'units': units}
        non_nan = sum(1 for v in params.values() if v is not None and not (isinstance(v,float) and np.isnan(v)))
        print(f'  {non_nan}/28 parameters computed')
        # Print key shape indices
        for k in ['DA_km2','Lp_km','Lb_km','Rff','Rc','Re','Dd','Rf','Ru','Sb','Elong']:
            v = params.get(k)
            u = units.get(k,'')
            print(f'  {k:12s}: {v:.3f} {u}' if isinstance(v,float) and not np.isnan(v) else f'  {k:12s}: NaN')
    except Exception as e:
        print(f'  ERROR: {e}')


In [ ]:
# Side-by-side geomorphic table
import warnings; warnings.filterwarnings('ignore')

dfs = []
for name, b in BASINS.items():
    if name in geo_params:
        s = pd.Series(geo_params[name]['params'], name=b['label'])
        dfs.append(s)

if dfs:
    df_geo = pd.concat(dfs, axis=1)
    df_geo = df_geo.applymap(lambda v: round(v,3) if isinstance(v,(int,float)) and not np.isnan(float(v)) else 'NaN')
    print('Geomorphic parameters — all three basins')
    df_geo


---
## 3. Topographic Wetness Index (TWI)

`compute_twi_result()` fetches the DEM (GLO-30), computes slope + flow accumulation, returns TWI statistics as a `HydroResult`.


In [ ]:
twi_results = {}

for name, b in BASINS.items():
    if name not in results:
        continue
    print(f"TWI — {b['label']}")
    try:
        twi_r = compute_twi_result(
            watershed_geojson=results[name].data['geometry_geojson'],
            resolution=30,
        )
        twi_results[name] = twi_r
        d = twi_r.data
        print(f"  TWI mean  : {d.get('twi_mean', float('nan')):.3f}")
        print(f"  TWI median: {d.get('twi_median', float('nan')):.3f}")
        print(f"  TWI std   : {d.get('twi_std', float('nan')):.3f}")
        print(f"  High wetness (TWI>10) pct: {d.get('high_wetness_pct', float('nan')):.1f}%")
    except Exception as e:
        print(f'  ⚠️  {e}')


---
## 4. Curve Number (CN)

Land cover from ESA WorldCover (Planetary Computer STAC) → remapped to NLCD codes. Soil hydrologic group from POLARIS (CONUS) or SoilGrids (global, GEE). CN lookup follows NRCS TR-55.


In [ ]:
cn_results = {}

for name, b in BASINS.items():
    if name not in wsheds:
        continue
    print(f"CN — {b['label']}")
    try:
        cn_r = create_curve_number_grid_from_geometry(
            watershed_gdf=wsheds[name],
            year=2020,
        )
        cn_results[name] = cn_r
        d = cn_r.data if hasattr(cn_r,'data') else cn_r
        cn_mean = d.get('cn_mean') or d.get('CN_weighted')
        print(f"  CN (weighted mean): {cn_mean:.1f}" if cn_mean else '  CN: not computed')
        for k in ['cn_mean','cn_std','pct_forest','pct_urban','pct_agriculture']:
            if k in d and d[k] is not None:
                print(f"  {k:25s}: {d[k]:.2f}")
    except Exception as e:
        print(f'  ⚠️  {e}')


---
## 5. Event runoff & design hydrograph (SCS-CN)

Analytical — no external data needed beyond CN. Uses SCS Curve Number method (NRCS TR-55).


In [ ]:
# Event runoff table — works with any CN, no live data needed
print('SCS-CN event runoff — design storms\n')

# Use CN from live computation if available, else representative defaults
BASIN_CN = {
    'potomac': cn_results.get('potomac', {}).get('data', {}).get('cn_mean') or 68.0,
    'thur'   : cn_results.get('thur',   {}).get('data', {}).get('cn_mean') or 65.0,
    'ping'   : cn_results.get('ping',   {}).get('data', {}).get('cn_mean') or 72.0,
}

design_storms_mm = [25, 50, 75, 100, 150]
rows = []
for name, b in BASINS.items():
    cn = BASIN_CN.get(name, 70.0)
    if hasattr(cn, '__float__'): cn = float(cn)
    S = storage_s_mm(cn)
    for P in design_storms_mm:
        Q = scs_cn_runoff(P, cn)
        rows.append({'Basin': b['label'].split('(')[0].strip(), 'CN': cn,
                     'P_mm': P, 'Q_mm': round(Q,1), 'S_mm': round(S,1),
                     'Q/P': round(Q/P,3) if P>0 else 0})

df_er = pd.DataFrame(rows)
print(df_er.to_string(index=False))


In [ ]:
# Design hydrograph — Thur River, 100-yr storm
cn_thur = BASIN_CN.get('thur', 65.0)
if hasattr(cn_thur,'__float__'): cn_thur = float(cn_thur)
area_thur = BASINS['thur']['area_km2']

# Time of concentration via Kirpich (main channel ~60 km, mean slope ~3%)
from aihydro_watershed.terrain.event_runoff import time_of_concentration_kirpich
tc_hr = time_of_concentration_kirpich(length_km=60.0, slope_m_per_m=0.03)
print(f'Tc (Kirpich) = {tc_hr:.2f} hr')

dh = design_hydrograph(area_km2=area_thur, tc_hr=tc_hr, cn=cn_thur, rainfall_mm=100.0)
t  = dh['time_hr']
q  = dh['discharge_cms']

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(t, q, alpha=0.3, color='#2196F3')
ax.plot(t, q, color='#1565C0', linewidth=2)
ax.axvline(dh['time_to_peak_hr'], color='red', linestyle='--', alpha=0.7, label=f"tp = {dh['time_to_peak_hr']:.1f} hr")
ax.axhline(dh['peak_discharge_cms'], color='orange', linestyle='--', alpha=0.7,
           label=f"Qp = {dh['peak_discharge_cms']:.0f} m³/s")
ax.set_xlabel('Time (hr)')
ax.set_ylabel('Discharge (m³/s)')
ax.set_title(f"Design hydrograph — {BASINS['thur']['label']}\nP=100 mm, CN={cn_thur:.0f}, Tc={tc_hr:.1f} hr")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Peak Q: {dh["peak_discharge_cms"]:.0f} m³/s  |  Runoff: {dh["runoff_mm"]:.1f} mm  |  Volume: {dh["runoff_volume_m3"]/1e6:.1f} M m³')


---
## 6. RUSLE erosion

`r_factor_renard_freimund()` from monthly precipitation · `k_factor_epic()` from soil texture · `ls_factor_wischmeier()` from slope/length · `c_factor_from_landcover()` from NLCD class. Fully analytical once inputs are assembled.


In [ ]:
# Representative basin characteristics (use live results where available)
BASIN_EROSION = {
    'potomac': dict(monthly_p=[55,60,85,90,100,95,100,90,85,80,65,58], slope_pct=6,  lc='mixed_forest',  sand_pct=45, silt_pct=35, clay_pct=20, oc_pct=2.5),
    'thur'   : dict(monthly_p=[60,55,65,75,90,95,90,85,75,65,60,58],  slope_pct=8,  lc='forest',        sand_pct=35, silt_pct=45, clay_pct=20, oc_pct=3.0),
    'ping'   : dict(monthly_p=[5,  8, 25,60,165,185,180,200,190,110,35,10], slope_pct=12, lc='crops', sand_pct=55, silt_pct=30, clay_pct=15, oc_pct=1.5),
}

print(f'{'Basin':<28}  {'R':>6}  {'K':>7}  {'LS':>5}  {'C':>6}  {'A (t/ha/yr)':>12}')
print('-'*75)
erosion_rows = []
for name, b in BASINS.items():
    e = BASIN_EROSION[name]
    R  = r_factor_renard_freimund(e['monthly_p'])
    K  = k_factor_epic(e['sand_pct'], e['silt_pct'], e['clay_pct'], e['oc_pct'])
    LS = ls_factor_wischmeier(e['slope_pct'], slope_length_m=100.0)
    C  = c_factor_from_landcover(e['lc'])
    A  = rusle(R, K, LS, C)
    erosion_rows.append({'basin': b['label'].split('(')[0].strip(), 'R':round(R,0),'K':round(K,4),'LS':round(LS,2),'C':round(C,4),'A':round(A,2)})
    print(f'{b["label"].split("(")[0].strip():<28}  {R:>6.0f}  {K:>7.4f}  {LS:>5.2f}  {C:>6.4f}  {A:>12.2f}')


---
## 7. Hydrological signatures — CAMELS-17

**Streamflow routing:**
- `gauge_id` supplied → USGS NWIS (CONUS)
- `gauge_id=None` → GEOGLOWS v2 (anonymous AWS S3 Zarr, 1940→present) → Open-Meteo GloFAS fallback

**Precipitation routing (for runoff_ratio / stream_elas):**  GridMET (CONUS) / ERA5-Land / CHIRPS (GEE) / CHIRPS-IRI OPeNDAP (no auth fallback)


In [ ]:
sig_results = {}

for name, b in BASINS.items():
    if name not in results:
        print(f'Skipping {name} (no delineation result)')
        continue
    print(f"\nSignatures — {b['label']}")
    print(f"  Streamflow source: {'USGS NWIS' if b['gauge_id'] else 'GEOGLOWS v2 (auto)'}")
    try:
        sig_r = extract_hydrological_signatures(
            gauge_id=b['gauge_id'],
            watershed_geojson=results[name].data['geometry_geojson'],
            area_km2=b['area_km2'],
            start_date=b['start_date'],
            end_date=b['end_date'],
        )
        sig_results[name] = sig_r
        sigs = sig_r.data
        for k in ['q_mean','q_median','baseflow_index','runoff_ratio','stream_elas',
                  'high_q_freq','low_q_freq','zero_q_freq','flow_variability',
                  'hfd_mean','slope_fdc']:
            v = sigs.get(k)
            flag = ' ⚠️' if v is None or (isinstance(v,float) and np.isnan(v)) else ''
            print(f"  {k:22s}: {round(v,4) if isinstance(v,(int,float)) and not np.isnan(float(v if v else 0)) else 'NaN'}{flag}")
    except Exception as e:
        print(f'  ⚠️  {e}')


In [ ]:
# Signature comparison bar chart
keys_plot = ['baseflow_index','runoff_ratio','flow_variability',
             'high_q_freq','low_q_freq','zero_q_freq']
labels = ['BFI','Runoff\nratio','Flow\nvar.','High Q\nfreq','Low Q\nfreq','Zero Q\nfreq']
colors = ['#2196F3','#4CAF50','#FF9800']
x = np.arange(len(keys_plot)); w = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, b) in enumerate(BASINS.items()):
    if name not in sig_results: continue
    sigs = sig_results[name].data
    vals = [sigs.get(k, np.nan) for k in keys_plot]
    vals = [v if v is not None else np.nan for v in vals]
    ax.bar(x + i*w, vals, w, label=b['label'].split('(')[0].strip(), color=colors[i], alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Value')
ax.set_title('CAMELS-style signatures — three global watersheds', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


---
## 8. Baseflow separation

Two filters: **Lyne-Hollick** (recursive, 3-pass) and **UKIH** (smoothed minima). The CAMELS signature tool uses Lyne-Hollick internally.

*Demonstrated here with synthetic discharge representing a tropical catchment. Pass any daily Q array.*


In [ ]:
# Synthetic Q — monsoon-driven catchment (use real Q from sig_results if available)
rng = np.random.default_rng(42)
n_days = 365 * 5
doy = np.arange(n_days) % 365
# Tropical monsoon: peak July-Oct
seasonal = 1.0 + 4.0 * np.exp(-0.5 * ((doy - 250) / 55)**2)
q_syn = (rng.lognormal(np.log(200), 0.7, n_days) * seasonal / seasonal.mean()).clip(2)
dates_syn = pd.date_range('2000-01-01', periods=n_days, freq='D')

bf_lh = lyne_hollick(q_syn, alpha=0.925, passes=3)
bf_uk = ukih(q_syn)
bfi_lh = compute_bfi(q_syn, bf_lh)
bfi_uk = compute_bfi(q_syn, bf_uk)

print(f'Lyne-Hollick BFI: {bfi_lh:.4f}')
print(f'UKIH BFI        : {bfi_uk:.4f}')

# Plot 1 year
sl = slice(0, 365)
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(dates_syn[sl], q_syn[sl], alpha=0.2, color='steelblue', label='Total flow')
ax.plot(dates_syn[sl], q_syn[sl], color='steelblue', linewidth=0.8)
ax.plot(dates_syn[sl], bf_lh[sl], color='#D32F2F', linewidth=1.5, label=f'Lyne-Hollick BFI={bfi_lh:.3f}')
ax.plot(dates_syn[sl], bf_uk[sl], color='#388E3C', linewidth=1.5, linestyle='--', label=f'UKIH BFI={bfi_uk:.3f}')
ax.set_xlabel('Date')
ax.set_ylabel('Discharge (m³/s)')
ax.set_title('Baseflow separation — tropical monsoon catchment (synthetic)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 9. Flow Duration Curve + 7Q10

FDC characterises the full range of river flows. 7Q10 is the 7-day low-flow threshold exceeded 90 % of years — a regulatory benchmark for water quality.


In [ ]:
q_pd = pd.Series(q_syn, index=dates_syn)  # from synthetic Q above

fdc   = flow_duration_curve(q_syn)
q7_10 = seven_q10(q_pd)

print(f"FDC slope  : {fdc['slope_fdc']:.4f}")
print(f"7Q10       : {q7_10['seven_q10']:.2f} m³/s  (over {q7_10['n_years']} years)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# FDC
ep = fdc['exceedance_prob']
qf = fdc['sorted_flows']
axes[0].semilogy(ep * 100, qf, color='steelblue', linewidth=2)
axes[0].axhline(q7_10['seven_q10'], color='red', linestyle='--',
                label=f"7Q10 = {q7_10['seven_q10']:.1f} m³/s")
axes[0].set_xlabel('Exceedance probability (%)')
axes[0].set_ylabel('Discharge (m³/s)')
axes[0].set_title('Flow Duration Curve')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

# Annual min 7-day flows
annual_7day = q7_10['annual_7day_min']
axes[1].bar(range(len(annual_7day)), sorted(annual_7day), color='steelblue', alpha=0.7)
axes[1].axhline(q7_10['seven_q10'], color='red', linestyle='--', label=f"7Q10 = {q7_10['seven_q10']:.1f}")
axes[1].set_xlabel('Year rank (sorted)')
axes[1].set_ylabel('7-day min Q (m³/s)')
axes[1].set_title('Annual 7-day minimum flows')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Flow Duration Curve & 7Q10', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 10. Flood frequency analysis

Fits a Gumbel EV-I distribution to annual peak flows via MLE. Returns return levels with 90 % bootstrap confidence intervals.


In [ ]:
am_result = annual_maxima(q_pd)   # dict with 'annual_max' list
am_vals   = am_result['annual_max']

ff = flood_frequency(am_vals, dist='gumbel', return_periods=[2,5,10,25,50,100], n_bootstrap=1000)

print(f"Distribution : {ff['dist']} ({ff['method']})")
print(f"Years of data: {ff['n_years']}")
print()
print(f"{'Return period':>15}  {'Q (m³/s)':>12}  {'CI low':>10}  {'CI high':>10}")
print('-' * 55)
for rl in ff['return_levels']:
    print(f"{rl['return_period']:>14.0f}y  {rl['value']:>12.1f}  {rl['ci_low']:>10.1f}  {rl['ci_high']:>10.1f}")

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
T_list = [r['return_period'] for r in ff['return_levels']]
Q_list = [r['value'] for r in ff['return_levels']]
ci_lo  = [r['ci_low'] for r in ff['return_levels']]
ci_hi  = [r['ci_high'] for r in ff['return_levels']]

ax.fill_between(T_list, ci_lo, ci_hi, alpha=0.2, color='steelblue', label='90% CI')
ax.semilogx(T_list, Q_list, 'o-', color='steelblue', linewidth=2, markersize=7, label='Gumbel fit')

# Empirical plotting positions
pp = ff['plotting_positions']
T_emp = [p['return_period'] for p in pp]
Q_emp = [p['value'] for p in pp]
ax.scatter(T_emp, Q_emp, color='#D32F2F', zorder=5, s=30, label='Observed annual max')

ax.set_xlabel('Return period (years)')
ax.set_ylabel('Peak discharge (m³/s)')
ax.set_title('Flood Frequency Analysis (Gumbel EV-I)')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()


---
## 11. Drought indices — SPI & SPEI

**SPI** (Standardised Precipitation Index) and **SPEI** (adds PET) at user-defined accumulation scales (3-month for seasonal, 12-month for multi-annual). Computed from monthly precipitation time series.


In [ ]:
# Synthetic monthly precipitation — 20 years with a dry period around year 10-13
rng2 = np.random.default_rng(99)
n_months = 240
dates_m = pd.date_range('2000-01', periods=n_months, freq='MS')
base_p = 80 + 30 * np.sin(np.arange(n_months) * 2 * np.pi / 12)  # seasonal cycle
# Drought: years 10-13 (months 120-156) get 40% less
base_p[120:156] *= 0.60
monthly_p  = list(np.maximum(base_p + rng2.normal(0, 15, n_months), 0))
monthly_pet = list(np.maximum(50 + 25 * np.sin(np.arange(n_months) * 2 * np.pi / 12)
                              + rng2.normal(0, 5, n_months), 5))

spi3  = spi(monthly_p,  dates=dates_m, scale_months=3)
spi12 = spi(monthly_p,  dates=dates_m, scale_months=12)
spei3 = spei(monthly_p, monthly_pet, dates=dates_m, scale_months=3)

spi3_arr  = np.array(spi3['index'],  dtype=float)
spi12_arr = np.array(spi12['index'], dtype=float)
spei3_arr = np.array(spei3['index'], dtype=float)
months    = pd.to_datetime(spi3['index_months'])

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

for ax, arr, label, scale in [
    (axes[0], spi3_arr,  'SPI-3',  3),
    (axes[1], spi12_arr, 'SPI-12', 12),
    (axes[2], spei3_arr, 'SPEI-3', 3),
]:
    ax.bar(months, arr, color=np.where(arr>=0, '#1976D2', '#D32F2F'), alpha=0.8, width=20)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(-1, color='orange', linewidth=1, linestyle='--', alpha=0.7, label='Moderate drought')
    ax.axhline(-2, color='red',    linewidth=1, linestyle='--', alpha=0.7, label='Severe drought')
    ax.set_ylabel(label)
    ax.set_ylim(-3.5, 3.5)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.2)
    # Shade drought period
    ax.axvspan(pd.Timestamp('2010-01-01'), pd.Timestamp('2013-01-01'),
               color='yellow', alpha=0.15, label='Induced drought')

axes[-1].set_xlabel('Date')
plt.suptitle('Drought indices — SPI-3, SPI-12, SPEI-3\n(induced drought 2010–2013 highlighted)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'SPI-3  drought months (< -1): {int(np.sum(spi3_arr < -1))} / {len(spi3_arr)}')
print(f'SPI-12 drought months (< -1): {int(np.sum(spi12_arr < -1))} / {len(spi12_arr)}')


---
## 12. Full watershed attribute vector

Assembling geomorphic + CN + signatures into a single CAMELS-style attribute table. This is the target output for `aihydro-lsh` (Wave C).


In [ ]:
summary_rows = []

for name, b in BASINS.items():
    row = {'basin': b['label']}

    # Delineation
    if name in results:
        row['area_km2'] = round(b['area_km2'], 1)

    # Geomorphic
    if name in geo_params:
        for k in ['DA_km2','Lb_km','Rff','Rc','Re','Dd','Rf','Sb','Elong','HKR']:
            v = geo_params[name]['params'].get(k)
            row[k] = round(v,3) if isinstance(v,float) and not np.isnan(v) else np.nan

    # CN
    if name in cn_results:
        d = cn_results[name].data if hasattr(cn_results[name],'data') else cn_results[name]
        row['CN_mean'] = round(d.get('cn_mean', np.nan), 1)

    # Signatures
    if name in sig_results:
        sigs = sig_results[name].data
        for k in ['q_mean','baseflow_index','runoff_ratio','stream_elas',
                  'high_q_freq','low_q_freq','flow_variability','slope_fdc']:
            v = sigs.get(k)
            row[k] = round(v,4) if isinstance(v,(int,float)) and not np.isnan(float(v if v else 'nan')) else np.nan

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('basin').T
print('Full watershed attribute vector')
df_summary


---
## Summary

All five `aihydro-watershed` capability domains demonstrated on three continents:

| Domain | Key functions | Global? |
|---|---|---|
| Delineation | `delineate_from_point()` | ✅ NLDI (CONUS) / MERIT-pyflwdir (global) |
| Geomorphic | `extract_geomorphic_parameters()` | ✅ GLO-30 DEM, geodesic area, UTM zone per basin |
| TWI | `compute_twi_result()` | ✅ Any DEM |
| Terrain | `create_curve_number_grid_from_geometry()` | ✅ ESA WorldCover STAC / SoilGrids (GEE) |
| Event runoff | `design_hydrograph()`, `scs_cn_runoff()` | ✅ analytical |
| Erosion | `rusle()`, `r_factor_renard_freimund()` | ✅ analytical |
| Signatures | `extract_hydrological_signatures()` | ✅ NWIS / GEOGLOWS / Open-Meteo |
| Baseflow | `lyne_hollick()`, `ukih()`, `compute_bfi()` | ✅ analytical |
| FDC | `flow_duration_curve()`, `seven_q10()` | ✅ analytical |
| Flood freq | `flood_frequency()`, `annual_maxima()` | ✅ analytical |
| Drought | `spi()`, `spei()` | ✅ analytical |

**Next step:** `aihydro-lsh` (Wave C) wraps this into a single `lsh.attributes(point)` call that returns all 71 CAMELS attributes for any watershed on Earth.
